In [116]:
import json
from collections import Counter

FILE = "realistic_events.jsonl"          # <-- change to your actual file path

records = [json.loads(line) for line in open(FILE, encoding="utf-8")]

print("total records:", len(records))

# 1) what does one raw event look like?
print("\n--- first record (all keys) ---")
print(json.dumps(records[0], indent=2))

# 2) what keys/fields exist across ALL events?
keys = set()
for r in records:
    keys.update(r.keys())
print("\nfields present:", sorted(keys))

# 3) what mix of event IDs + channels is in the file?
mix = Counter((r.get("channel"), r.get("event_id")) for r in records)
print("\n(channel, event_id) counts:")
for (ch, eid), n in sorted(mix.items(), key=lambda kv: str(kv[0])):
    print(f"  {ch:10} {eid:>6} : {n}")

total records: 1096

--- first record (all keys) ---
{
  "timestamp": "2026-08-01T08:52:00.000Z",
  "channel": "Security",
  "provider": "Microsoft-Windows-Security-Auditing",
  "event_id": 4624,
  "computer": "WIN-WS01",
  "user": "alice",
  "level": "Information",
  "task": "None",
  "opcode": "Info",
  "record_id": 514002,
  "message": "An account was successfully logged on.",
  "SubjectUserName": "alice",
  "TargetUserName": "alice",
  "TargetDomainName": "CORP",
  "TargetUserSid": "S-1-5-21-1000000000-1000000000-1000000000-1101",
  "LogonType": 2,
  "LogonId": "0x52e6b438",
  "AuthenticationPackageName": "Kerberos",
  "LogonProcessName": "User32",
  "IpAddress": "10.10.10.21",
  "IpPort": 50734,
  "WorkstationName": "WIN-WS01",
  "ElevatedToken": "No",
  "VirtualAccount": "No",
  "RestrictedAdminMode": "No"
}

fields present: ['AccessMask', 'AuthenticationPackageName', 'CommandLine', 'Company', 'CreationUtcTime', 'CurrentDirectory', 'Description', 'DestinationHostname', 'Destinati

In [117]:
import json
from collections import Counter

FILE = "data.txt"          # <-- your JSONL file

records = [json.loads(line) for line in open(FILE, encoding="utf-8")]

print("total records:", len(records))

# 1) first record, all keys
print("\n--- first record (all keys) ---")
print(json.dumps(records[0], indent=2))

# 2) fields present across ALL events
keys = set()
for r in records:
    keys.update(r.keys())
print("\nfields present:", sorted(keys))

# 3) (channel, event_id) mix
mix = Counter((r.get("channel"), r.get("event_id")) for r in records)
print("\n(channel, event_id) counts:")
for (ch, eid), n in sorted(mix.items(), key=lambda kv: (kv[0], kv[1])):
    print(f"  {ch:38} {eid:>5} : {n}")

# 4) mapping-critical fields, coverage per key event type
CRITICAL = ["LogonId", "ProcessGuid", "User", "user", "TargetUserSid",
            "TargetUserName", "IpAddress", "WorkstationName", "computer", "timestamp"]
print("\n--- mapping-critical fields (coverage per event type) ---")
for eid in [4624,4768,4769,5,2,7,10,11, 4625, 1, 3, 22]:
    rows = [r for r in records if r["event_id"] == eid]
    if not rows:
        continue
    cov = {f: sum(1 for r in rows if r.get(f)) for f in CRITICAL}
    print(f"event {eid}: total={len(rows)}")
    print("   " + ", ".join(f"{f}={cov[f]}" for f in CRITICAL))

total records: 4152

--- first record (all keys) ---
{
  "timestamp": "2026-08-01T00:20:49.972Z",
  "channel": "Microsoft-Windows-Sysmon/Operational",
  "provider": "Microsoft-Windows-Sysmon",
  "event_id": 1,
  "computer": "WIN-WS04",
  "user": "diana",
  "level": "Information",
  "task": "None",
  "opcode": "Info",
  "record_id": 871389,
  "event_sequence": 58,
  "activity_id": "0e572cd6-db0c-4d57-b844-8b5da172b120",
  "provider_guid": "44488f99-4e0b-49be-be83-e9daca4db68f",
  "message": "Process Create.",
  "UtcTime": "2026-08-01T00:20:49.972Z",
  "ProcessGuid": "8647d870-a539-4bfa-9ad5-e31b8a5d6c42",
  "ProcessId": 48605,
  "Image": "C:\\Windows\\System32\\RuntimeBroker.exe",
  "FileVersion": "10.0.26100.1",
  "Description": "RuntimeBroker.exe",
  "Product": "Microsoft Windows",
  "Company": "Microsoft Corporation",
  "CommandLine": "\"C:\\Windows\\System32\\RuntimeBroker.exe\"",
  "CurrentDirectory": "C:\\Windows\\System32",
  "User": "CORP\\diana",
  "LogonId": "0xfc4b91bf",
  "T

In [118]:
import json
from collections import Counter

FILE = "data.txt"          # <-- your JSONL file

# ── the keep-list: every event ID we care about ──
KEEP_EVENTS = [
    # Security
    4624, 4625, 4634, 4647, 4688, 4768, 4769, 4771, 4776,
    4698, 4724, 4738, 5140, 5145,
    # Sysmon
    1, 3, 5, 7, 10, 11, 12, 13, 15, 17, 18, 22, 23, 24,
]

# ── load all lines ──
records = [json.loads(line) for line in open(FILE, encoding="utf-8")]
print("total records before filter:", len(records))

# ── drop events not in KEEP_EVENTS ──
dropped = [r for r in records if r["event_id"] not in KEEP_EVENTS]
records = [r for r in records if r["event_id"] in KEEP_EVENTS]
print("kept  :", len(records))
print("dropped:", len(dropped))

# ── what got dropped, and why ──
drop_mix = Counter((r.get("channel"), r.get("event_id")) for r in dropped)
if drop_mix:
    print("\n(dropped channel, event_id) counts:")
    for (ch, eid), n in sorted(drop_mix.items(), key=lambda kv: (kv[0], kv[1])):
        print(f"  {ch:38} {eid:>5} : {n}")

# ── what's left, per (channel, event_id) ──
mix = Counter((r.get("channel"), r.get("event_id")) for r in records)
print("\n(kept channel, event_id) counts:")
for (ch, eid), n in sorted(mix.items(), key=lambda kv: (kv[0], kv[1])):
    print(f"  {ch:38} {eid:>5} : {n}")

total records before filter: 4152
kept  : 4152
dropped: 0

(kept channel, event_id) counts:
  Microsoft-Windows-Sysmon/Operational       1 : 554
  Microsoft-Windows-Sysmon/Operational       3 : 1013
  Microsoft-Windows-Sysmon/Operational       5 : 16
  Microsoft-Windows-Sysmon/Operational       7 : 96
  Microsoft-Windows-Sysmon/Operational      10 : 28
  Microsoft-Windows-Sysmon/Operational      11 : 330
  Microsoft-Windows-Sysmon/Operational      12 : 81
  Microsoft-Windows-Sysmon/Operational      13 : 81
  Microsoft-Windows-Sysmon/Operational      15 : 82
  Microsoft-Windows-Sysmon/Operational      17 : 17
  Microsoft-Windows-Sysmon/Operational      18 : 17
  Microsoft-Windows-Sysmon/Operational      22 : 879
  Microsoft-Windows-Sysmon/Operational      23 : 23
  Microsoft-Windows-Sysmon/Operational      24 : 15
  Security                                4624 : 44
  Security                                4625 : 12
  Security                                4634 : 40
  Security         

In [119]:
# ── Stage 1b: extract important fields into ONE clean normalized record ──

def strip_domain(u):
    return u.split("\\")[-1] if u and "\\" in u else u


def normalize(r):
    if "host" in r and "computer" not in r:
        return r

    eid = r["event_id"]
    raw_user = (r.get("User") or r.get("TargetUserName") or r.get("SubjectUserName") or r.get("user"))
    
    return {
        "host": r.get("computer"), "time": r.get("timestamp"), "event_id": eid,
        "user": strip_domain(raw_user),
        "sid": r.get("TargetUserSid") or r.get("SubjectUserSid"),
        "logon_id": r.get("LogonId") or r.get("TargetLogonId") or r.get("SubjectLogonId"),
        "process_guid": r.get("ProcessGuid"),
        "parent_process_guid": r.get("ParentProcessGuid"),
        "image": r.get("Image"), "cmdline": r.get("CommandLine"),
        "parent_image": r.get("ParentImage"), "parent_user": r.get("ParentUser"),
        "dst_ip": r.get("DestinationIp"), "dst_port": r.get("DestinationPort"),
        "src_ip": r.get("SourceIp") or r.get("IpAddress"), "initiated": r.get("Initiated"),
        "workstation": r.get("WorkstationName") or r.get("Workstation"),
        "logon_type": r.get("LogonType"), "target_user": r.get("TargetUserName"),
        "query_name": r.get("QueryName"), "target_filename": r.get("TargetFilename"),
        "target_object": r.get("TargetObject"), "target_image": r.get("TargetImage"),
        "target_process_guid": r.get("TargetProcessGUID"), "granted_access": r.get("GrantedAccess"),
        "pipe_name": r.get("PipeName"), "image_loaded": r.get("ImageLoaded"),
        "service_name": r.get("ServiceName"), "share_name": r.get("ShareName"),
        "relative_target": r.get("RelativeTargetName"), "status": r.get("Status"),
    }

records = [normalize(r) for r in records]
print("records re-normalized:", len(records))

print("normalized records:", len(records))
for eid in (4624, 1, 3, 22):
    r = next(r for r in records if r["event_id"] == eid)
    print(f"\n--- normalized event {eid} ---")
    for k, v in r.items():
        if v is not None:
            print(f"  {k:14} = {v}")

records re-normalized: 4152
normalized records: 4152

--- normalized event 4624 ---
  host           = WIN-WS03
  time           = 2026-08-01T07:52:48.180Z
  event_id       = 4624
  user           = charlie
  sid            = S-1-5-21-1000000000-1000000000-1000000000-1103
  logon_id       = 0xd7e60db7
  src_ip         = 10.10.10.23
  workstation    = WIN-WS03
  logon_type     = 2
  target_user    = charlie

--- normalized event 1 ---
  host           = WIN-WS04
  time           = 2026-08-01T00:20:49.972Z
  event_id       = 1
  user           = diana
  logon_id       = 0xfc4b91bf
  process_guid   = 8647d870-a539-4bfa-9ad5-e31b8a5d6c42
  parent_process_guid = a740e9e3-4c59-4602-a6ea-3143b63f5799
  image          = C:\Windows\System32\RuntimeBroker.exe
  cmdline        = "C:\Windows\System32\RuntimeBroker.exe"
  parent_image   = C:\Windows\System32\services.exe
  parent_user    = CORP\diana

--- normalized event 3 ---
  host           = WIN-WS04
  time           = 2026-08-01T03:15:20.846Z

In [98]:
raw_user = r.get("User") or r.get("TargetUserName") or r.get("SubjectUserName") or r.get("user")
logon_id = r.get("LogonId") or r.get("TargetLogonId") or r.get("SubjectLogonId")

In [120]:
# ── Stage 2a: PASS 1 — build GLOBAL maps from the WHOLE dataset ──

logon_map = {}   # (host, logon_id) -> session info from Security 4624
proc_map  = {}   # process_guid    -> process info from Sysmon 1

for r in records:
    if r["event_id"] == 4624:
        logon_map[(r["host"], r["logon_id"])] = {
            "user": r["user"], "sid": r["sid"],
            "src_ip": r["src_ip"], "workstation": r["workstation"],
            "logon_type": r["logon_type"], "time": r["time"]}
    elif r["event_id"] == 1:
        proc_map[r["process_guid"]] = {
            "user": r["user"], "logon_id": r["logon_id"],
            "image": r["image"], "time": r["time"]}

print("logon_map entries:", len(logon_map))
print("proc_map entries :", len(proc_map))
print("\nexample logon_map entry:")
print(" ", next(iter(logon_map.items())))
print("\nexample proc_map entry:")
print(" ", next(iter(proc_map.items())))

logon_map entries: 44
proc_map entries : 554

example logon_map entry:
  (('WIN-WS03', '0xd7e60db7'), {'user': 'charlie', 'sid': 'S-1-5-21-1000000000-1000000000-1000000000-1103', 'src_ip': '10.10.10.23', 'workstation': 'WIN-WS03', 'logon_type': 2, 'time': '2026-08-01T07:52:48.180Z'})

example proc_map entry:
  ('8647d870-a539-4bfa-9ad5-e31b8a5d6c42', {'user': 'diana', 'logon_id': '0xfc4b91bf', 'image': 'C:\\Windows\\System32\\RuntimeBroker.exe', 'time': '2026-08-01T00:20:49.972Z'})


In [121]:
# ── Stage 2b: PASS 2 — resolve user + enrich with login context ──

def session_of(r):
    """Find the 4624 session that owns this event, if any."""
    p = proc_map.get(r["process_guid"])          # 1) via process_guid
    lid = r["logon_id"] or (p or {}).get("logon_id")
    return logon_map.get((r["host"], lid))        # 2) via (host, logon_id)

def resolve_user(r):
    # 1) direct identity (Sysmon User / Security TargetUserName / SubjectUserName / top-level user)
    if r["user"] and not r["user"].startswith("NT AUTHORITY"):
        return r["user"], r["sid"]
    # 2) chain: process -> its logon session -> the account that logged on
    sess = session_of(r)
    if sess:
        return sess["user"], sess["sid"]
    return None, None

for r in records:
    r["resolved_user"], r["resolved_sid"] = resolve_user(r)
    sess = session_of(r)                          # enrichment (the mapping payoff)
    if sess:
        r["login_ip"]         = sess["src_ip"]
        r["login_type"]       = sess["logon_type"]
        r["login_workstation"] = sess["workstation"]
    else:
        r["login_ip"] = r["login_type"] = r["login_workstation"] = None

# ── show what mapping+enrichment achieved on one event of each kind ──
print(f"{'event':>6} {'resolved_user':14} {'own field?':9} {'login_ip':12} {'login_type':11} {'workstation'}")
for eid in (4624, 1, 3, 22, 5145):
    r = next(r for r in records if r["event_id"] == eid)
    own = "direct" if r["user"] else "chained"
    print(f"{eid:>6} {str(r['resolved_user']):14} {own:9} {str(r['login_ip']):12} {str(r['login_type']):11} {r['login_workstation']}")

print("\nunattributed events:", sum(1 for r in records if r["resolved_user"] is None))
print("events now carrying a login_ip :", sum(1 for r in records if r["login_ip"]))

 event resolved_user  own field? login_ip     login_type  workstation
  4624 charlie        direct    10.10.10.23  2           WIN-WS03
     1 diana          direct    None         None        None
     3 diana          direct    None         None        None
    22 charlie        direct    None         None        None
  5145 diana          direct    None         None        None

unattributed events: 0
events now carrying a login_ip : 468


In [122]:
# ── Stage 2a: PASS 1 — build GLOBAL maps from the WHOLE dataset ──

logon_map = {}   # (host, logon_id) -> session info from Security 4624
proc_map  = {}   # process_guid    -> process info from Sysmon 1

for r in records:
    if r["event_id"] == 4624:
        logon_map[(r["host"], r["logon_id"])] = {
            "user": r["user"], "sid": r["sid"],
            "src_ip": r["src_ip"], "workstation": r["workstation"],
            "logon_type": r["logon_type"], "time": r["time"]}
    elif r["event_id"] == 1:
        proc_map[r["process_guid"]] = {
            "user": r["user"], "logon_id": r["logon_id"],
            "image": r["image"], "time": r["time"]}

print("logon_map entries:", len(logon_map))
print("proc_map entries :", len(proc_map))

logon_map entries: 44
proc_map entries : 554


In [123]:
# ── Stage 2b: PASS 2 — resolve user + enrich with login context ──

def session_of(r):
    p = proc_map.get(r["process_guid"])          # 1) via process_guid
    lid = r["logon_id"] or (p or {}).get("logon_id")
    return logon_map.get((r["host"], lid))        # 2) via (host, logon_id)

def resolve_user(r):
    if r["user"] and not r["user"].startswith("NT AUTHORITY"):
        return r["user"], r["sid"]                # direct identity
    sess = session_of(r)                          # else: chain
    if sess:
        return sess["user"], sess["sid"]
    return None, None

for r in records:
    r["resolved_user"], r["resolved_sid"] = resolve_user(r)
    sess = session_of(r)                          # enrichment (the mapping payoff)
    if sess:
        r["login_ip"]          = sess["src_ip"]
        r["login_type"]        = sess["logon_type"]
        r["login_workstation"] = sess["workstation"]
        r["login_sid"]         = sess["sid"]
    else:
        r["login_ip"] = r["login_type"] = r["login_workstation"] = r["login_sid"] = None

# ── SAMPLE OUTPUT from the REAL data ──

examples = [
    ("a real 4624",                        next(r for r in records if r["event_id"] == 4624)),
    ("a real Sysmon 3",                    next(r for r in records if r["event_id"] == 3 and r["user"])),
    ("blanked Sysmon 3 (user removed)",    None),   # filled below
]
blanked = dict(next(r for r in records if r["event_id"] == 3 and r["user"]))
blanked["user"] = None                              # simulate a missing user field
examples[2] = ("blanked Sysmon 3 (user removed)", blanked)

print("========== BEFORE ==========")
for label, e in examples:
    print(f"{label:34} | event {e['event_id']} | host={e['host']} | "
          f"user={str(e['user']):8} | process_guid={str(e['process_guid'])[:13]}")

print("\n========== AFTER ==========")
for label, e in examples:
    u, sid = resolve_user(e)
    sess = session_of(e)
    if e["user"] and not e["user"].startswith("NT AUTHORITY"):
        why = "DIRECT (has user)"
    elif sess:
        why = "CHAIN recovered"
    else:
        why = "CHAIN failed (no matching 4624)"
    print(f"{label}:  [{why}]")
    print(f"    resolved_user  = {u}")
    print(f"    login_ip       = {sess['src_ip'] if sess else None}")
    print(f"    login_type     = {sess['logon_type'] if sess else None}")
    print()

# ── SUMMARY ──
print("========== SUMMARY ==========")
print("records processed :", len(records))
print("unattributed      :", sum(1 for r in records if r["resolved_user"] is None))
print("got login_ip      :", sum(1 for r in records if r["login_ip"]))

========== BEFORE ==========
a real 4624                        | event 4624 | host=WIN-WS03 | user=charlie  | process_guid=None
a real Sysmon 3                    | event 3 | host=WIN-WS04 | user=diana    | process_guid=07829951-3552
blanked Sysmon 3 (user removed)    | event 3 | host=WIN-WS04 | user=None     | process_guid=07829951-3552

========== AFTER ==========
a real 4624:  [DIRECT (has user)]
    resolved_user  = charlie
    login_ip       = 10.10.10.23
    login_type     = 2

a real Sysmon 3:  [DIRECT (has user)]
    resolved_user  = diana
    login_ip       = None
    login_type     = None

blanked Sysmon 3 (user removed):  [CHAIN failed (no matching 4624)]
    resolved_user  = None
    login_ip       = None
    login_type     = None

========== SUMMARY ==========
records processed : 4152
unattributed      : 0
got login_ip      : 468


In [124]:
# ── Stage 3 — bucket into (user, 10-min slot) ──
from datetime import datetime, timezone
from collections import defaultdict, Counter

WINDOW_SEC = 600

def slot_of(ts):
    dt = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    return int(dt.timestamp()) // WINDOW_SEC * WINDOW_SEC

def slot_label(s):
    return datetime.fromtimestamp(s, tz=timezone.utc).strftime("%m-%d %H:%M")

buckets = defaultdict(list)
for r in records:
    buckets[(r["resolved_user"] or "UNATTRIBUTED", slot_of(r["time"]))].append(r)

print("distinct buckets:", len(buckets))

# sample: one user's day
u = "diana"
day1_end = slot_of("2026-08-02T00:00:00Z")
for (user, s), rs in sorted(buckets.items()):
    if user == u and s < day1_end:
        ids = Counter(r["event_id"] for r in rs)
        print(f"  {slot_label(s)}  -> {len(rs):2d} events  {dict(ids)}")

distinct buckets: 1259
  08-01 00:20  ->  1 events  {1: 1}
  08-01 03:10  ->  2 events  {3: 1, 22: 1}
  08-01 04:00  ->  1 events  {22: 1}
  08-01 06:10  ->  3 events  {3: 1, 5140: 1, 5145: 1}
  08-01 07:40  ->  1 events  {22: 1}
  08-01 07:50  ->  3 events  {4624: 1, 4768: 1, 4769: 1}
  08-01 08:30  ->  1 events  {1: 1}
  08-01 09:10  ->  1 events  {3: 1}
  08-01 09:30  ->  1 events  {11: 1}
  08-01 10:00  ->  3 events  {22: 1, 12: 1, 13: 1}
  08-01 10:50  ->  3 events  {3: 1, 5140: 1, 5145: 1}
  08-01 12:10  ->  1 events  {11: 1}
  08-01 12:30  -> 10 events  {4688: 1, 1: 1, 22: 4, 3: 4}
  08-01 12:40  ->  5 events  {3: 1, 5140: 1, 5145: 1, 12: 1, 13: 1}
  08-01 12:50  ->  9 events  {4688: 1, 1: 1, 22: 5, 3: 2}
  08-01 13:00  ->  8 events  {4698: 1, 4688: 1, 1: 1, 7: 1, 11: 1, 22: 1, 12: 1, 13: 1}
  08-01 13:40  -> 11 events  {17: 1, 18: 1, 4688: 1, 1: 1, 3: 3, 15: 1, 22: 2, 11: 1}
  08-01 13:50  ->  1 events  {1: 1}
  08-01 14:00  ->  1 events  {3: 1}
  08-01 14:50  ->  4 events  {46

In [125]:
# ── Stage 4 — feature vector per bucket ──

def count_id(rs, eid): return sum(1 for r in rs if r["event_id"] == eid)
def nunique(rs, eid, field):
    return len({r[field] for r in rs if r["event_id"] == eid and r[field] is not None})

FEATURES = []
# B1 volume — auto counter for EVERY kept event ID (nothing forgotten)
for eid in KEEP_EVENTS:
    FEATURES.append((f"n_{eid}", lambda rs, eid=eid: count_id(rs, eid)))

# B2 cardinality — the distinct-value signals
FEATURES += [
    ("nuniq_3_DestinationIp",   lambda rs: nunique(rs, 3, "dst_ip")),
    ("nuniq_3_DestinationPort", lambda rs: nunique(rs, 3, "dst_port")),
    ("nuniq_1_Image",           lambda rs: nunique(rs, 1, "image")),
    ("nuniq_22_QueryName",      lambda rs: nunique(rs, 22, "query_name")),
    ("nuniq_11_TargetFilename", lambda rs: nunique(rs, 11, "target_filename")),
    ("nuniq_4624_WorkstationName", lambda rs: nunique(rs, 4624, "workstation")),
    ("nuniq_4624_IpAddress",    lambda rs: nunique(rs, 4624, "src_ip")),
    ("active_processes",        lambda rs: len({r["process_guid"] for r in rs if r["process_guid"]})),
]

# B3 presence — rare critical events: fire on ANY occurrence
PRESENCE_EVENTS = [4698, 4724, 4738, 4771, 24]
for eid in PRESENCE_EVENTS:
    FEATURES.append((f"has_{eid}", lambda rs, eid=eid: 1 if count_id(rs, eid) else 0))

def feature_vector(rs):
    return {name: fn(rs) for name, fn in FEATURES}

# sample: busiest bucket for one user
key, rs = max(((k, v) for k, v in buckets.items() if k[0] == "diana"), key=lambda kv: len(kv[1]))
vec = feature_vector(rs)
print("busiest diana bucket:", slot_label(key[1]), f"({len(rs)} events)")
print({k: v for k, v in vec.items() if v})
print("total feature columns:", len(FEATURES))

busiest diana bucket: 08-02 12:20 (22 events)
{'n_4688': 2, 'n_1': 2, 'n_3': 9, 'n_11': 1, 'n_15': 1, 'n_22': 7, 'nuniq_3_DestinationIp': 6, 'nuniq_3_DestinationPort': 1, 'nuniq_1_Image': 1, 'nuniq_22_QueryName': 5, 'nuniq_11_TargetFilename': 1, 'active_processes': 13}
total feature columns: 41


In [126]:
from statistics import mean, pstdev
from math import log2

LOLBINS = {"certutil.exe","rundll32.exe","powershell.exe","powershell_ise.exe","mshta.exe",
           "wscript.exe","cscript.exe","regsvr32.exe","cmd.exe","wmic.exe","bitsadmin.exe",
           "schtasks.exe","reg.exe","whoami.exe","net.exe","net1.exe","nltest.exe",
           "msbuild.exe","csc.exe","installutil.exe","regasm.exe","cmstp.exe","bash.exe","psexec.exe"}

SHELLS = {"cmd.exe","powershell.exe","powershell_ise.exe","wscript.exe","cscript.exe","bash.exe","mshta.exe"}

SUSPICIOUS_PAIRS = {
    ("winword.exe","powershell.exe"), ("winword.exe","cmd.exe"),
    ("excel.exe","powershell.exe"), ("excel.exe","cmd.exe"),
    ("outlook.exe","powershell.exe"), ("outlook.exe","cmd.exe"),
    ("msedge.exe","powershell.exe"), ("msedge.exe","cmd.exe"),
    ("chrome.exe","powershell.exe"), ("chrome.exe","cmd.exe"),
    ("wscript.exe","powershell.exe"), ("mshta.exe","cmd.exe"),
    ("explorer.exe","mshta.exe"),
}

def base_name(path): return (path or "").lower().split("\\")[-1]

def is_lolbin(r): return base_name(r.get("image")) in LOLBINS

def lolbin_path_anomaly(r):
    if not is_lolbin(r): return False
    p = (r.get("image") or "").lower()
    return ("system32" not in p) and ("syswow64" not in p)

def suspicious_parent_child(r):
    return (base_name(r.get("parent_image")), base_name(r.get("image"))) in SUSPICIOUS_PAIRS

def parent_user_mismatch(r):
    pu, u = r.get("parent_user") or "", r.get("user") or ""
    return bool(pu and u and strip_domain(pu) != strip_domain(u))

def special_char_density(s):
    if not s: return 0.0
    return sum(1 for c in s if not c.isalnum()) / len(s)

def shannon_entropy(s):
    if not s: return 0.0
    cnt = Counter(s); n = len(s)
    return -sum((c/n) * log2(c/n) for c in cnt.values())

def is_external(ip):
    if not ip: return False
    return not (ip.startswith(("10.","192.168.","172.16.")) or ip == "127.0.0.1")

def has_encoded(s):
    if not s: return False
    ls = s.lower()
    return any(k in ls for k in ("-enc", "-e ", "frombase64string", " -w hidden", "bypass"))

print("helpers ready —", len(LOLBINS), "LOLBins,", len(SUSPICIOUS_PAIRS), "suspicious pairs")

helpers ready — 24 LOLBins, 13 suspicious pairs


In [127]:
def epoch(ts): return int(datetime.fromisoformat(ts.replace("Z","+00:00")).timestamp())

proc_start, proc_end = {}, {}
proc_image, proc_cmdline = {}, {}
proc_parent, proc_parent_image, proc_parent_user, proc_user = {}, {}, {}, {}
conns_by_proc = defaultdict(list)      # pg -> [(epoch, dst_ip, dst_port)]
downloads_by_proc = Counter()

for r in records:
    pg = r.get("process_guid")
    if not pg: continue
    if r["event_id"] == 1:
        proc_start[pg] = epoch(r["time"]); proc_image[pg] = r.get("image")
        proc_cmdline[pg] = r.get("cmdline"); proc_parent[pg] = r.get("parent_process_guid")
        proc_parent_image[pg] = r.get("parent_image"); proc_parent_user[pg] = r.get("parent_user")
        proc_user[pg] = r.get("user")
    elif r["event_id"] == 5:  proc_end[pg] = epoch(r["time"])
    elif r["event_id"] == 3:  conns_by_proc[pg].append((epoch(r["time"]), r.get("dst_ip"), r.get("dst_port")))
    elif r["event_id"] == 15: downloads_by_proc[pg] += 1

def beacon_regularity(pg):
    """1.0 = perfectly regular intervals (beacon). None if too few intervals."""
    by_ip = defaultdict(list)
    for t, ip, port in conns_by_proc[pg]: by_ip[ip].append(t)
    best = None
    for ip, ts in by_ip.items():
        if len(ts) < 4: continue
        gaps = [b - a for a, b in zip(sorted(ts), sorted(ts)[1:])]
        if mean(gaps) <= 0: continue
        reg = 1.0 - min(1.0, pstdev(gaps) / mean(gaps))
        best = reg if best is None else max(best, reg)
    return best

def _pum(pu, u): return bool(pu and u and strip_domain(pu) != strip_domain(u))

proc_feats = {}
for pg in proc_start:
    lifetime = proc_end.get(pg, proc_start[pg]) - proc_start[pg]
    img = proc_image.get(pg) or ""; cmdl = proc_cmdline.get(pg) or ""
    parent_pg = proc_parent.get(pg)
    proc_feats[pg] = {
        "unique_ips": len({c[1] for c in conns_by_proc[pg]}),
        "downloads": downloads_by_proc.get(pg, 0),
        "cmd_char_length": len(cmdl),
        "special_char_density": special_char_density(cmdl),
        "is_lolbin": base_name(img) in LOLBINS,
        "lolbin_path_anomaly": base_name(img) in LOLBINS and
                               ("system32" not in img.lower() and "syswow64" not in img.lower()),
        "suspicious_parent_child": (base_name(proc_parent_image.get(pg)), base_name(img)) in SUSPICIOUS_PAIRS,
        "parent_user_mismatch": _pum(proc_parent_user.get(pg, ""), proc_user.get(pg, "")),
        "lifetime_seconds": lifetime,
        "is_micro_duration": lifetime < 5,
        "parent_dead_before_spawn": bool(parent_pg and parent_pg in proc_end
                                         and proc_end[parent_pg] < proc_start[pg]),
        "beacon_regularity": beacon_regularity(pg),
    }

print("processes tracked:", len(proc_start), "| with Event5 end:", len(proc_end))

processes tracked: 554 | with Event5 end: 16


In [128]:
seen = defaultdict(set)
for r in records:
    u = r["user"] or "?"
    if r.get("dst_ip"):      seen[(u, "ip")].add(r["dst_ip"])
    if r.get("query_name"):  seen[(u, "domain")].add(r["query_name"])
    if r.get("image"):       seen[(u, "image")].add(r["image"])
    if r.get("cmdline"):     seen[(u, "cmdline")].add(r["cmdline"])
    if r.get("workstation"): seen[(u, "ws")].add(r["workstation"])
    if r.get("dst_port") and r.get("image"):
        seen[(u, "imgport")].add((base_name(r["image"]), r["dst_port"]))

print("seen unique values (per user):")
for u in ("diana", "bob"):
    print(f"  {u}: {len(seen[(u,'ip')])} IPs, {len(seen[(u,'domain')])} domains, "
          f"{len(seen[(u,'image')])} images, {len(seen[(u,'imgport')])} (image,port) pairs")

seen unique values (per user):
  diana: 12 IPs, 8 domains, 14 images, 21 (image,port) pairs
  bob: 12 IPs, 8 domains, 14 images, 18 (image,port) pairs


In [129]:
def count_id(rs, eid): return sum(1 for r in rs if r["event_id"] == eid)
def nunique(rs, eid, field):
    return len({r[field] for r in rs if r["event_id"] == eid and r.get(field) is not None})

def window_features(rs):
    u = rs[0]["user"] if rs else None
    n_4625 = count_id(rs, 4625); n_4624 = count_id(rs, 4624)
    conns = [r for r in rs if r["event_id"] == 3]
    ext = [r for r in conns if is_external(r.get("dst_ip"))]
    inbound = [r for r in conns if r.get("initiated") in (False, "false", "False")]
    pgs = {r.get("process_guid") for r in rs if r.get("process_guid")}
    pf = [proc_feats[p] for p in pgs if p in proc_feats]

    f = {}
    # B1 volume — every kept event ID
    for eid in KEEP_EVENTS: f[f"n_{eid}"] = count_id(rs, eid)
    # B2 cardinality
    f["nuniq_3_DestinationIp"] = nunique(rs, 3, "dst_ip")
    f["nuniq_3_DestinationPort"] = nunique(rs, 3, "dst_port")
    f["nuniq_1_Image"] = nunique(rs, 1, "image")
    f["nuniq_22_QueryName"] = nunique(rs, 22, "query_name")
    f["nuniq_11_TargetFilename"] = nunique(rs, 11, "target_filename")
    f["nuniq_4624_WorkstationName"] = nunique(rs, 4624, "workstation")
    f["nuniq_4625_TargetUserName"] = nunique(rs, 4625, "target_user")
    f["active_processes"] = len(pgs)
    # B3 novelty (the stars)
    f["new_dst_ips"] = sum(1 for r in conns if r.get("dst_ip") and r.get("dst_ip") not in seen[(u, "ip")])
    f["new_domains"] = sum(1 for r in rs if r["event_id"] == 22 and r.get("query_name")
                           and r.get("query_name") not in seen[(u, "domain")])
    f["new_images"] = sum(1 for r in rs if r["event_id"] == 1 and r.get("image")
                          and r.get("image") not in seen[(u, "image")])
    f["new_cmdlines"] = sum(1 for r in rs if r["event_id"] == 1 and r.get("cmdline")
                            and r.get("cmdline") not in seen[(u, "cmdline")])
    f["new_workstations"] = sum(1 for r in rs if r["event_id"] == 4624 and r.get("workstation")
                                and r.get("workstation") not in seen[(u, "ws")])
    # B4 ratios
    f["failed_logon_ratio"] = n_4625 / (n_4625 + n_4624) if (n_4625 + n_4624) else 0.0
    f["external_ip_ratio"] = len(ext) / len(conns) if conns else 0.0
    f["inbound_ratio"] = len(inbound) / len(conns) if conns else 0.0
    # B5 semantic (record-level)
    f["n_lolbin"] = sum(1 for r in rs if r["event_id"] == 1 and is_lolbin(r))
    f["n_lolbin_path_anomaly"] = sum(1 for r in rs if r["event_id"] == 1 and lolbin_path_anomaly(r))
    f["n_suspicious_parent_child"] = sum(1 for r in rs if r["event_id"] == 1 and suspicious_parent_child(r))
    f["n_parent_user_mismatch"] = sum(1 for r in rs if r["event_id"] == 1 and parent_user_mismatch(r))
    f["has_encoded_cmd"] = 1 if any(has_encoded(r.get("cmdline")) for r in rs if r["event_id"] == 1) else 0
    f["lsass_access"] = 1 if any(r["event_id"] == 10 and base_name(r.get("target_image")) == "lsass.exe" for r in rs) else 0
    f["runkey_write"] = 1 if any(r["event_id"] in (12, 13) and r.get("target_object")
                                 and "run" in r["target_object"].lower() for r in rs) else 0
    f["sensitive_share"] = 1 if any(r["event_id"] == 5145 and r.get("share_name")
                                    and r["share_name"].lower().rstrip("\\").endswith(("admin$", "c$")) for r in rs) else 0
    f["is_shell_network"] = 1 if any(base_name(r.get("image")) in SHELLS for r in conns) else 0
    f["internal_445_anomaly"] = sum(1 for r in conns if r.get("dst_port") == 445
                                    and not is_external(r.get("dst_ip"))
                                    and base_name(r.get("image")) not in ("explorer.exe", "svchost.exe"))
    f["unusual_port_for_process"] = sum(1 for r in conns if r.get("dst_port")
                                        and (base_name(r.get("image")), r.get("dst_port")) not in seen[(u, "imgport")])
    f["dns_entropy_max"] = max([shannon_entropy(r.get("query_name")) for r in rs if r["event_id"] == 22], default=0.0)
    f["smb_pipe_activity"] = 1 if (any(r.get("dst_port") == 445 for r in conns) and any(r["event_id"] == 17 for r in rs)) else 0
    # B6 presence
    for eid in [4698, 4724, 4738, 4771, 24]: f[f"has_{eid}"] = 1 if count_id(rs, eid) else 0
    f["has_4776_fail"] = 1 if any(r["event_id"] == 4776 and str(r.get("status")) != "0x0" for r in rs) else 0
    # B7 process-derived
    f["n_micro_duration_procs"] = sum(1 for p in pf if p["is_micro_duration"])
    f["n_parent_dead_spawn"] = sum(1 for p in pf if p["parent_dead_before_spawn"])
    f["n_downloads"] = sum(p["downloads"] for p in pf)
    f["max_beacon_regularity"] = max([p["beacon_regularity"] for p in pf if p["beacon_regularity"] is not None], default=0.0)
    f["max_cmd_char_length"] = max([p["cmd_char_length"] for p in pf], default=0)
    f["max_special_char_density"] = max([p["special_char_density"] for p in pf], default=0.0)
    return f

In [130]:
def slot_of(ts): return int(datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp()) // 600 * 600
def slot_label(s): return datetime.fromtimestamp(s, tz=timezone.utc).strftime("%m-%d %H:%M")

# build buckets from records (self-contained — no need for the Stage 3 cell)
buckets = defaultdict(list)
for r in records:
    buckets[(r["user"] or "UNATTRIBUTED", slot_of(r["time"]))].append(r)

# pick the busiest window for a sample user
key, rs = max(((k, v) for k, v in buckets.items() if k[0] == "diana"),
              key=lambda kv: len(kv[1]))
vec = window_features(rs)

print("INPUT  : one bucket (user, 10-min slot)")
print(f"  user = {key[0]}   slot = {slot_label(key[1])}   raw events = {len(rs)}")
print("  events by ID:", dict(Counter(r['event_id'] for r in rs)))
print()
print("OUTPUT : one feature vector (non-zero features only)")
for k, v in vec.items():
    if v:
        print(f"  {k:32} = {v}")
print(f"  ... {len(vec) - sum(1 for v in vec.values() if v)} more features = 0 (omitted)")
print(f"  total feature columns = {len(vec)}")

INPUT  : one bucket (user, 10-min slot)
  user = diana   slot = 08-02 12:20   raw events = 22
  events by ID: {4688: 2, 1: 2, 22: 7, 3: 9, 11: 1, 15: 1}

OUTPUT : one feature vector (non-zero features only)
  n_4688                           = 2
  n_1                              = 2
  n_3                              = 9
  n_11                             = 1
  n_15                             = 1
  n_22                             = 7
  nuniq_3_DestinationIp            = 6
  nuniq_3_DestinationPort          = 1
  nuniq_1_Image                    = 1
  nuniq_22_QueryName               = 5
  nuniq_11_TargetFilename          = 1
  active_processes                 = 13
  external_ip_ratio                = 1.0
  dns_entropy_max                  = 3.327819531114783
  n_micro_duration_procs           = 2
  max_cmd_char_length              = 83
  max_special_char_density         = 0.18072289156626506
  ... 52 more features = 0 (omitted)
  total feature columns = 69


In [131]:
USER = "diana"    # <-- change user
HOUR = 13         # <-- change hour (0-23)

def slot_of(ts): return int(datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp()) // 600 * 600
def day_of(s): return datetime.fromtimestamp(s, tz=timezone.utc).strftime("%m-%d")

buckets = defaultdict(list)
for r in records:
    buckets[(r["user"] or "UNATTRIBUTED", slot_of(r["time"]))].append(r)

days = sorted({day_of(s) for (u, s) in buckets if u == USER})

# compute the feature vector for all 6 slots x days of this hour
windows = []
for d in days:
    for i in range(6):
        dt = datetime(2026, 8, int(d[:2]), HOUR, i * 10, tzinfo=timezone.utc)
        slot = int(dt.timestamp())
        vec = window_features(buckets.get((USER, slot), []))   # [] -> all-zero vector
        windows.append((d, i * 10, vec))

# only columns that are non-zero in AT LEAST one window (drops the 51 always-zero columns)
active_cols = [k for k in windows[0][2] if any(w[2][k] for w in windows)]

print(f"FULL feature vectors for {USER}, hour {HOUR:02d}  "
      f"({len(active_cols)} active of {len(windows[0][2])} features)")
print()
hdr = "  day  slot"
for c in active_cols:
    hdr += f"  {c:>9}"
print(hdr)

for d, m, vec in windows:
    row = f"  {d} {HOUR:02d}:{m:02d}"
    for c in active_cols:
        v = vec[c]
        row += f"  {v:8.2f}" if isinstance(v, float) else f"  {v:>9}"
    print(row)

FULL feature vectors for diana, hour 13  (18 active of 69 features)

  day  slot     n_4688        n_1        n_3       n_11       n_15       n_22       n_23  nuniq_3_DestinationIp  nuniq_3_DestinationPort  nuniq_1_Image  nuniq_22_QueryName  nuniq_11_TargetFilename  active_processes  external_ip_ratio  dns_entropy_max  n_micro_duration_procs  max_cmd_char_length  max_special_char_density
  08-01 13:00          0          0          0          1          0          0          1          0          0          0          0          1          2      0.00      0.00          0          0      0.00
  08-01 13:10          0          0          0          0          0          1          0          0          0          0          1          0          0      0.00      3.09          0          0      0.00
  08-01 13:20          1          1          3          1          1          3          0          2          1          1          3          1          6      1.00      3.48          1    

In [133]:
def slot_of(ts): return int(datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp()) // 600 * 600
def slot_label(s): return datetime.fromtimestamp(s, tz=timezone.utc).strftime("%m-%d %H:%M")

buckets = defaultdict(list)
for r in records:
    buckets[(r["user"] or "UNATTRIBUTED", slot_of(r["time"]))].append(r)

# --- NORMAL window: diana's busiest ---
key, rs = max(((k, v) for k, v in buckets.items() if k[0] == "diana"), key=lambda kv: len(kv[1]))
normal = window_features(rs)
print("NORMAL  diana", slot_label(key[1]), f"({len(rs)} events) — non-zero features:")
print({k: v for k, v in normal.items() if v})

# --- ANOMALY window: injected (encoded PowerShell, port 4444, lsass, DGA) ---
anom = [
    dict(time="2026-08-10T02:00:00Z", event_id=1, user="diana", process_guid="{A1}", parent_process_guid="{A0}",
         image="C:\\Users\\Public\\powershell.exe", cmdline="powershell.exe -enc JABzAD0...",
         parent_image="C:\\Program Files\\Microsoft Office\\root\\Office16\\WINWORD.EXE", parent_user="CORP\\diana"),
    dict(time="2026-08-10T02:00:05Z", event_id=3, user="diana", process_guid="{A1}",
         image="C:\\Users\\Public\\powershell.exe", dst_ip="203.0.113.99", dst_port=4444, initiated=True),
    dict(time="2026-08-10T02:00:10Z", event_id=3, user="diana", process_guid="{A1}",
         image="C:\\Users\\Public\\powershell.exe", dst_ip="203.0.113.99", dst_port=4444, initiated=True),
    dict(time="2026-08-10T02:00:15Z", event_id=3, user="diana", process_guid="{A1}",
         image="C:\\Users\\Public\\powershell.exe", dst_ip="203.0.113.99", dst_port=4444, initiated=True),
    dict(time="2026-08-10T02:00:20Z", event_id=3, user="diana", process_guid="{A1}",
         image="C:\\Users\\Public\\powershell.exe", dst_ip="203.0.113.99", dst_port=4444, initiated=True),
    dict(time="2026-08-10T02:00:11Z", event_id=10, user="diana", process_guid="{A1}",
         image="C:\\Users\\Public\\powershell.exe", target_image="C:\\Windows\\System32\\lsass.exe"),
    dict(time="2026-08-10T02:00:25Z", event_id=22, user="diana", query_name="x7k2m9q-v4n8.xyz"),
    dict(time="2026-08-10T02:00:40Z", event_id=4625, user="diana", target_user="diana", src_ip="203.0.113.7", workstation="WIN-WS04"),
    dict(time="2026-08-10T02:00:45Z", event_id=4625, user="diana", target_user="diana", src_ip="203.0.113.7", workstation="WIN-WS04"),
]

# register the injected process so process-derived features can see it
proc_start["{A1}"] = epoch(anom[0]["time"]); proc_image["{A1}"] = anom[0]["image"]
proc_cmdline["{A1}"] = anom[0]["cmdline"]; proc_parent["{A1}"] = "{A0}"
proc_parent_image["{A1}"] = anom[0]["parent_image"]; proc_parent_user["{A1}"] = "CORP\\diana"
proc_user["{A1}"] = "diana"
conns_by_proc["{A1}"] = [(epoch(r["time"]), r.get("dst_ip"), r.get("dst_port")) for r in anom if r["event_id"] == 3]
proc_feats["{A1}"] = dict(
    unique_ips=1, downloads=0, cmd_char_length=len(anom[0]["cmdline"]),
    special_char_density=special_char_density(anom[0]["cmdline"]),
    is_lolbin=True, lolbin_path_anomaly=True, suspicious_parent_child=True,
    parent_user_mismatch=False, lifetime_seconds=45, is_micro_duration=False,
    parent_dead_before_spawn=False, beacon_regularity=beacon_regularity("{A1}"))

anomaly = window_features(anom)
print("\nANOMALY diana (9 injected events) — non-zero features:")
print({k: v for k, v in anomaly.items() if v})

print("\n=== features that LIT UP (zero in normal, non-zero in anomaly) ===")
for k in anomaly:
    if anomaly[k] and not normal.get(k):
        print(f"  {k:30} normal={normal.get(k, 0):>6}  anomaly={anomaly[k]}")

NORMAL  diana 08-02 12:20 (22 events) — non-zero features:
{'n_4688': 2, 'n_1': 2, 'n_3': 9, 'n_11': 1, 'n_15': 1, 'n_22': 7, 'nuniq_3_DestinationIp': 6, 'nuniq_3_DestinationPort': 1, 'nuniq_1_Image': 1, 'nuniq_22_QueryName': 5, 'nuniq_11_TargetFilename': 1, 'active_processes': 13, 'external_ip_ratio': 1.0, 'dns_entropy_max': 3.327819531114783, 'n_micro_duration_procs': 2, 'max_cmd_char_length': 83, 'max_special_char_density': 0.18072289156626506}

ANOMALY diana (9 injected events) — non-zero features:
{'n_4625': 2, 'n_1': 1, 'n_3': 4, 'n_10': 1, 'n_22': 1, 'nuniq_3_DestinationIp': 1, 'nuniq_3_DestinationPort': 1, 'nuniq_1_Image': 1, 'nuniq_22_QueryName': 1, 'nuniq_4625_TargetUserName': 1, 'active_processes': 1, 'new_dst_ips': 4, 'new_domains': 1, 'new_images': 1, 'new_cmdlines': 1, 'failed_logon_ratio': 1.0, 'external_ip_ratio': 1.0, 'n_lolbin': 1, 'n_lolbin_path_anomaly': 1, 'n_suspicious_parent_child': 1, 'has_encoded_cmd': 1, 'lsass_access': 1, 'is_shell_network': 1, 'unusual_port_

In [134]:
# ── Stage 5a: zero-filled grid (user, day, slot) -> feature vector ──
from statistics import median

def slot_of(ts): return int(datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp()) // 600 * 600
def day_of(s):   return datetime.fromtimestamp(s, tz=timezone.utc).strftime("%Y-%m-%d")
def hour_of(s):  return datetime.fromtimestamp(s, tz=timezone.utc).hour

buckets = defaultdict(list)
for r in records:
    buckets[(r["user"] or "UNATTRIBUTED", slot_of(r["time"]))].append(r)

users = sorted({r["user"] for r in records if r["user"]})
days  = sorted({day_of(slot_of(r["time"])) for r in records})

grid = {}
for u in users:
    for d in days:
        day_start = int(datetime.strptime(d, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())
        for i in range(144):                            # every 10-min slot of the day
            slot = day_start + i * 600
            grid[(u, d, slot)] = window_features(buckets.get((u, slot), []))   # [] -> all zeros

print("grid cells:", len(grid), "=", len(users), "users x", len(days), "days x 144 slots")

grid cells: 5760 = 4 users x 10 days x 144 slots


In [135]:
# ── Stage 5a: zero-filled grid (user, day, slot) -> feature vector ──
from statistics import median

def slot_of(ts): return int(datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp()) // 600 * 600
def day_of(s):   return datetime.fromtimestamp(s, tz=timezone.utc).strftime("%Y-%m-%d")
def hour_of(s):  return datetime.fromtimestamp(s, tz=timezone.utc).hour

buckets = defaultdict(list)
for r in records:
    buckets[(r["user"] or "UNATTRIBUTED", slot_of(r["time"]))].append(r)

users = sorted({r["user"] for r in records if r["user"]})
days  = sorted({day_of(slot_of(r["time"])) for r in records})

grid = {}
for u in users:
    for d in days:
        day_start = int(datetime.strptime(d, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())
        for i in range(144):                            # every 10-min slot of the day
            slot = day_start + i * 600
            grid[(u, d, slot)] = window_features(buckets.get((u, slot), []))   # [] -> all zeros

print("grid cells:", len(grid), "=", len(users), "users x", len(days), "days x 144 slots")

grid cells: 5760 = 4 users x 10 days x 144 slots


In [ ]:
# ── Stage 5c: print the baseline (self-contained) ──
from statistics import median

def mad(vals):
    m = median(vals)
    return median([abs(v - m) for v in vals]) or 0.5

# rebuild baseline if this cell is run in a fresh kernel
if "baseline" not in dir() or not baseline:
    from collections import defaultdict
    def slot_of(ts): return int(datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp()) // 600 * 600
    def day_of(s):   return datetime.fromtimestamp(s, tz=timezone.utc).strftime("%Y-%m-%d")
    def hour_of(s):  return datetime.fromtimestamp(s, tz=timezone.utc).hour

    buckets = defaultdict(list)
    for r in records:
        buckets[(r["user"] or "UNATTRIBUTED", slot_of(r["time"]))].append(r)

    users = sorted({r["user"] for r in records if r["user"]})
    days  = sorted({day_of(slot_of(r["time"])) for r in records})

    grid = {}
    for u in users:
        for d in days:
            day_start = int(datetime.strptime(d, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())
            for i in range(144):
                slot = day_start + i * 600
                grid[(u, d, slot)] = window_features(buckets.get((u, slot), []))

    baseline = defaultdict(lambda: defaultdict(list))
    for (u, d, slot), vec in grid.items():
        baseline[u][hour_of(slot)].append(vec)

def active_count(vec):
    return sum(v for k, v in vec.items() if k.startswith("n_"))

sample = next(iter(grid.values()))
MEASURE_KEYS = [f for f in sample
                if f.startswith(("n_", "nuniq_", "active_", "external_",
                                 "failed_", "inbound_", "max_", "dns_"))]

def print_baseline(u=None, h=None):
    us = [u] if u else users
    for uname in us:
        hours = [h] if h is not None else sorted(baseline[uname])
        for hh in hours:
            pool = baseline[uname][hh]
            n_act = sum(1 for v in pool if active_count(v) > 0)
            print(f"=== {uname}  hour {hh:02d}:00–{hh:02d}:59  |  "
                  f"{len(pool)} windows | {n_act:2d} active | {len(pool) - n_act:2d} empty ===")
            sig = [(f, median([v[f] for v in pool])) for f in MEASURE_KEYS if any(v[f] for v in pool)]
            for f, m in sig:
                print(f"    {f:28} median={m:6.2f}  mad={mad([v[f] for v in pool]):6.2f}")
            if not sig:
                print("    (no activity in this hour — all measure features 0)")
        print()

print("BASELINE STRUCTURE:", {u: len(baseline[u]) for u in users}, "hours each")
print()
print_baseline("alice", 9)

BASELINE STRUCTURE: {'alice': 24, 'bob': 24, 'charlie': 24, 'diana': 24} hours each

=== bob  hour 13:00–13:59  |  60 windows | 23 active | 37 empty ===
    n_4624                       median=  0.00  mad=  0.50
    n_4688                       median=  0.00  mad=  0.50
    n_4768                       median=  0.00  mad=  0.50
    n_5140                       median=  0.00  mad=  0.50
    n_5145                       median=  0.00  mad=  0.50
    n_1                          median=  0.00  mad=  0.50
    n_3                          median=  0.00  mad=  0.50
    n_7                          median=  0.00  mad=  0.50
    n_11                         median=  0.00  mad=  0.50
    n_12                         median=  0.00  mad=  0.50
    n_13                         median=  0.00  mad=  0.50
    n_15                         median=  0.00  mad=  0.50
    n_17                         median=  0.00  mad=  0.50
    n_18                         median=  0.00  mad=  0.50
    n_22             